<a href="https://colab.research.google.com/github/saiKelkar/Product_Oriented_Scan-to-CAD_Dimensional_Inspection_System/blob/main/scan_to_cad_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install open3d

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/Scan-to-CAD_System')

folders = [
    "data/nominal",
    "data/synthetic",
    "data/real",
    "src",
    "configs",
    "outputs/reports",
    "outputs/visualizations",
    "benchmarks"
]

for folder in folders:
  (PROJECT_ROOT / folder).mkdir(
      parents=True,
      exist_ok=True
  )

print("Project folders created!")

for folder in folders:
  print(PROJECT_ROOT / folder)

Project folders created!
/content/drive/MyDrive/Scan-to-CAD_System/data/nominal
/content/drive/MyDrive/Scan-to-CAD_System/data/synthetic
/content/drive/MyDrive/Scan-to-CAD_System/data/real
/content/drive/MyDrive/Scan-to-CAD_System/src
/content/drive/MyDrive/Scan-to-CAD_System/configs
/content/drive/MyDrive/Scan-to-CAD_System/outputs/reports
/content/drive/MyDrive/Scan-to-CAD_System/outputs/visualizations
/content/drive/MyDrive/Scan-to-CAD_System/benchmarks


In [ ]:
CAD_PATH = (
    PROJECT_ROOT /
    "data/nominal/bracket.stl"
)

SCAN_PATH = (
    PROJECT_ROOT /
    "data/synthetic/raw_scan.ply"
)

GROUND_TRUTH_PATH = (
    PROJECT_ROOT /
    "data/synthetic/ground_truth.json"
)

CONFIG_PATH = (
    PROJECT_ROOT /
    "configs/inspection_config.json"
)

for name, path in [
    ("Nominal CAD", CAD_PATH),
    ("Raw scan", SCAN_PATH),
    ("Ground truth", GROUND_TRUTH_PATH),
    ("Configuration", CONFIG_PATH)
]:
    print(
        f"{name}:",
        "FOUND" if path.exists() else "MISSING"
    )

Nominal CAD: FOUND
Raw scan: FOUND
Ground truth: FOUND
Configuration: FOUND


In [ ]:
import open3d as o3d

mesh = o3d.io.read_triangle_mesh(CAD_PATH)
source = o3d.io.read_point_cloud(SCAN_PATH)

In [ ]:
%%writefile /content/geometry.py

from pathlib import Path

import numpy as np
import open3d as o3d


def load_cad(path):
    """Load and validate a nominal CAD triangle mesh."""

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            f"CAD file not found: {path}"
        )

    mesh = o3d.io.read_triangle_mesh(str(path))

    if mesh.is_empty() or len(mesh.triangles) == 0:
        raise ValueError("CAD mesh is empty or invalid.")

    vertices = np.asarray(mesh.vertices)

    if not np.all(np.isfinite(vertices)):
        raise ValueError(
            "CAD contains non-finite vertex coordinates."
        )

    mesh.compute_triangle_normals()
    mesh.compute_vertex_normals()

    return mesh


def load_scan(path):
    """Load and validate a 3D point cloud."""

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            f"Scan file not found: {path}"
        )

    scan = o3d.io.read_point_cloud(str(path))

    if scan.is_empty():
        raise ValueError("Scan point cloud is empty.")

    points = np.asarray(scan.points)

    if not np.all(np.isfinite(points)):
        raise ValueError(
            "Scan contains non-finite coordinates."
        )

    return scan


def get_geometry_info(geometry):
    """Return basic geometric information."""

    bbox = geometry.get_axis_aligned_bounding_box()

    info = {
        "dimensions": bbox.get_extent().tolist(),
        "min_bound": bbox.get_min_bound().tolist(),
        "max_bound": bbox.get_max_bound().tolist()
    }

    if isinstance(geometry, o3d.geometry.TriangleMesh):
        info["vertices"] = len(geometry.vertices)
        info["triangles"] = len(geometry.triangles)

    elif isinstance(geometry, o3d.geometry.PointCloud):
        info["points"] = len(geometry.points)

    return info

Writing /content/geometry.py


In [ ]:
import shutil

GEOMETRY_MODULE = PROJECT_ROOT / "src/geometry.py"

GEOMETRY_MODULE.parent.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy(
    "/content/geometry.py",
    GEOMETRY_MODULE
)

print("Module saved:", GEOMETRY_MODULE)

Module saved: /content/drive/MyDrive/Scan-to-CAD_System/src/geometry.py


In [ ]:
import sys

SRC_PATH = str(PROJECT_ROOT / "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from geometry import (
    load_cad,
    load_scan,
    get_geometry_info
)

In [ ]:
CAD_PATH = PROJECT_ROOT / "data/nominal/bracket.stl"
SCAN_PATH = PROJECT_ROOT / "data/synthetic/raw_scan.ply"

nominal_mesh = load_cad(CAD_PATH)
raw_scan = load_scan(SCAN_PATH)

print("CAD loaded successfully!")
print("Scan loaded successfully!")

CAD loaded successfully!
Scan loaded successfully!


In [ ]:
cad_info = get_geometry_info(nominal_mesh)
scan_info = get_geometry_info(raw_scan)

print("CAD information:")
print(cad_info)

print("\nRaw scan information:")
print(scan_info)

CAD information:
{'dimensions': [15.0, 49.5, 49.5], 'min_bound': [-1.0, -1.0, 0.0], 'max_bound': [14.0, 48.5, 49.5], 'vertices': 2128, 'triangles': 1140}

Raw scan information:
{'dimensions': [43.6898011690686, 58.43016341717748, 59.260623717404386], 'min_bound': [-5.697667182620879, 29.383679092872583, 10.094953994838518], 'max_bound': [37.992133986447726, 87.81384251005007, 69.3555777122429], 'points': 9122}


In [ ]:
try:
    load_cad("nonexistent_bracket.stl")

except FileNotFoundError as e:
    print("Expected error caught:")
    print(e)

Expected error caught:
CAD file not found: nonexistent_bracket.stl


In [ ]:
from geometry import load_cad, load_scan, get_geometry_info

nominal_mesh = load_cad(CAD_PATH)
raw_scan = load_scan(SCAN_PATH)

print("CAD loaded successfully!")
print("Scan loaded successfully!")

print("\nCAD information:")
print(get_geometry_info(nominal_mesh))

print("\nRaw scan information:")
print(get_geometry_info(raw_scan))

CAD loaded successfully!
Scan loaded successfully!

CAD information:
{'dimensions': [15.0, 49.5, 49.5], 'min_bound': [-1.0, -1.0, 0.0], 'max_bound': [14.0, 48.5, 49.5], 'vertices': 2128, 'triangles': 1140}

Raw scan information:
{'dimensions': [43.6898011690686, 58.43016341717748, 59.260623717404386], 'min_bound': [-5.697667182620879, 29.383679092872583, 10.094953994838518], 'max_bound': [37.992133986447726, 87.81384251005007, 69.3555777122429], 'points': 9122}


In [15]:
registration_code = '''
import copy
import numpy as np
import open3d as o3d


def register_scan(nominal_mesh, raw_scan, config):
    """
    Align raw scan to nominal CAD.

    Returns:
        registered_scan
        diagnostics
    """

    cfg = config["registration"]
    voxel = cfg["voxel_size_mm"]

    # 1. Reference point cloud
    target = nominal_mesh.sample_points_poisson_disk(
        number_of_points=10000
    )

    source = copy.deepcopy(raw_scan)

    # 2. Downsample
    source_down = source.voxel_down_sample(voxel)
    target_down = target.voxel_down_sample(voxel)

    # 3. Estimate normals
    normal_radius = voxel * cfg["normal_radius_multiplier"]

    for cloud in [source_down, target_down]:
        cloud.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(
                radius=normal_radius,
                max_nn=30
            )
        )

    # 4. Compute FPFH
    feature_radius = voxel * cfg["fpfh_radius_multiplier"]

    def compute_fpfh(cloud):
        return o3d.pipelines.registration.compute_fpfh_feature(
            cloud,
            o3d.geometry.KDTreeSearchParamHybrid(
                radius=feature_radius,
                max_nn=100
            )
        )

    source_fpfh = compute_fpfh(source_down)
    target_fpfh = compute_fpfh(target_down)

    # 5. RANSAC coarse registration
    distance = voxel * cfg["ransac_distance_multiplier"]

    ransac = (
        o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
            source_down,
            target_down,
            source_fpfh,
            target_fpfh,
            mutual_filter=True,
            max_correspondence_distance=distance,
            estimation_method=(
                o3d.pipelines.registration.TransformationEstimationPointToPoint(
                    False
                )
            ),
            ransac_n=3,
            checkers=[
                o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(
                    0.9
                ),
                o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(
                    distance
                )
            ],
            criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(
                100000, 0.999
            )
        )
    )

    # 6. Full ICP
    full_icp = o3d.pipelines.registration.registration_icp(
        source,
        target,
        cfg["icp_distance_mm"],
        ransac.transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        o3d.pipelines.registration.ICPConvergenceCriteria(
            max_iteration=100
        )
    )

    # 7. Transform a temporary source for stable-region selection
    source_coarse = copy.deepcopy(source)
    source_coarse.transform(full_icp.transformation)

    source_points = np.asarray(source_coarse.points)
    target_points = np.asarray(target.points)

    y_max = cfg["stable_region_y_max_mm"]

    source_indices = np.where(source_points[:, 1] < y_max)[0]
    target_indices = np.where(target_points[:, 1] < y_max)[0]

    source_stable = source_coarse.select_by_index(
        source_indices.tolist()
    )

    target_stable = target.select_by_index(
        target_indices.tolist()
    )

    # 8. Stable-region ICP
    stable_icp = o3d.pipelines.registration.registration_icp(
        source_stable,
        target_stable,
        cfg["icp_distance_mm"],
        np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        o3d.pipelines.registration.ICPConvergenceCriteria(
            max_iteration=100
        )
    )

    # Compose transformations
    final_transform = (
        stable_icp.transformation @ full_icp.transformation
    )

    # 9. Apply to original scan
    registered_scan = copy.deepcopy(raw_scan)
    registered_scan.transform(final_transform)

    # 10. Evaluate final alignment against the same reference
    final_eval = o3d.pipelines.registration.evaluate_registration(
        raw_scan,
        target,
        cfg["icp_distance_mm"],
        final_transform
    )

    diagnostics = {
        "ransac_fitness": float(ransac.fitness),
        "ransac_rmse_mm": float(ransac.inlier_rmse),
        "full_icp_fitness": float(full_icp.fitness),
        "full_icp_rmse_mm": float(full_icp.inlier_rmse),
        "stable_icp_fitness": float(stable_icp.fitness),
        "stable_icp_rmse_mm": float(stable_icp.inlier_rmse),
        "final_fitness": float(final_eval.fitness),
        "final_rmse_mm": float(final_eval.inlier_rmse),
        "stable_source_points": int(len(source_stable.points)),
        "stable_target_points": int(len(target_stable.points)),
        "transformation": final_transform.tolist()
    }

    return registered_scan, diagnostics
'''

REGISTRATION_PATH = PROJECT_ROOT / "src/registration.py"
REGISTRATION_PATH.parent.mkdir(parents=True, exist_ok=True)
REGISTRATION_PATH.write_text(registration_code)

print("Saved:", REGISTRATION_PATH)

Saved: /content/drive/MyDrive/Scan-to-CAD_System/src/registration.py


In [17]:
import json

CONFIG_PATH = PROJECT_ROOT / "configs/inspection_config.json"

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

print("Configuration loaded!")

Configuration loaded!


In [22]:
# Add missing registration parameters
defaults = {
    "voxel_size_mm": 1.0,
    "normal_radius_multiplier": 2.0,
    "fpfh_radius_multiplier": 5.0,
    "ransac_distance_multiplier": 1.5,
    "icp_distance_mm": 3.0,
    "stable_region_y_max_mm": 25.0
}

config.setdefault("registration", {})

for key, value in defaults.items():
    config["registration"].setdefault(key, value)

# Save the updated configuration
with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

print("Registration configuration updated!")

print(json.dumps(config["registration"], indent=2))

Registration configuration updated!
{
  "voxel_size_mm": 1.0,
  "normal_radius_multiplier": 2.0,
  "fpfh_radius_multiplieir": 5.0,
  "ransac_distance_multiplier": 1.5,
  "icp_distance_mm": 3.0,
  "stable_region_y_max_mm": 25.0,
  "fpfh_radius_multiplier": 5.0
}


In [23]:
import sys
import importlib

SRC_PATH = str(PROJECT_ROOT / "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import registration
importlib.reload(registration)

registered_scan, diagnostics = registration.register_scan(
    nominal_mesh,
    raw_scan,
    config
)

print("Registration completed!")

for key, value in diagnostics.items():
    if key != "transformation":
        print(f"{key}: {value}")

[Open3D WARNING] Too few correspondences (173) after mutual filter, fall back to original correspondences.
Registration completed!
ransac_fitness: 0.6493144317917732
ransac_rmse_mm: 0.767126658806296
full_icp_fitness: 0.9868449901337426
full_icp_rmse_mm: 0.5214522826512565
stable_icp_fitness: 0.99185667752443
stable_icp_rmse_mm: 0.3343146041379651
final_fitness: 0.986625739969305
final_rmse_mm: 0.6811592710826426
stable_source_points: 6754
stable_target_points: 7413


In [24]:
from scipy.spatial.transform import Rotation
import numpy as np

GROUND_TRUTH_PATH = (
    PROJECT_ROOT / "data/synthetic/ground_truth.json"
)

with open(GROUND_TRUTH_PATH, "r") as f:
    ground_truth = json.load(f)

T_true = np.array(
    ground_truth["scan_pose"]["scan_to_cad_transformation"]
)

T_est = np.array(diagnostics["transformation"])

R_delta = T_est[:3, :3] @ T_true[:3, :3].T

rotation_error = np.degrees(
    Rotation.from_matrix(R_delta).magnitude()
)

translation_error = np.linalg.norm(
    T_est[:3, 3] - T_true[:3, 3]
)

print(f"Rotation error: {rotation_error:.4f} degrees")
print(f"Translation error: {translation_error:.4f} mm")

Rotation error: 0.3104 degrees
Translation error: 0.1713 mm


In [25]:
def check_registration(diagnostics, config):
    min_fitness = config["quality"]["minimum_registration_fitness"]

    if diagnostics["final_fitness"] < min_fitness:
        return "REVIEW"

    if not np.isfinite(diagnostics["final_rmse_mm"]):
        return "REVIEW"

    if diagnostics["stable_source_points"] < 100:
        return "REVIEW"

    return "ACCEPTABLE"


registration_status = check_registration(diagnostics, config)

print("Registration status:", registration_status)

Registration status: ACCEPTABLE


In [26]:
config["registration"].pop(
    "fpfh_radius_multiplieir",
    None
)

with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

print("Configuration cleaned!")

Configuration cleaned!


In [27]:
features_code = '''
import numpy as np
from scipy.optimize import least_squares


def measure_horizontal_hole(
    points,
    approx_y,
    approx_x,
    search_radius=3.5,
    z_min=1.2,
    z_max=3.2,
    nominal_radius=2.5
):
    """
    Measure a Z-axis cylindrical hole.

    Returns a measurement dictionary.
    Returns REVIEW if fitting is unreliable.
    """

    points = np.asarray(points)

    radial = np.sqrt(
        (points[:, 1] - approx_y)**2 +
        (points[:, 0] - approx_x)**2
    )

    mask = (
        (radial < search_radius) &
        (points[:, 2] > z_min) &
        (points[:, 2] < z_max)
    )

    wall_points = points[mask]

    if len(wall_points) < 50:
        return {
            "status": "REVIEW",
            "reason": "Insufficient candidate wall points",
            "num_points": int(len(wall_points))
        }

    xy = wall_points[:, [1, 0]]

    def residuals(params):
        cy, cx, radius = params

        distances = np.sqrt(
            (xy[:, 0] - cy)**2 +
            (xy[:, 1] - cx)**2
        )

        return distances - radius

    result = least_squares(
        residuals,
        x0=[approx_y, approx_x, nominal_radius],
        bounds=(
            [approx_y - 1.0, approx_x - 1.0, 2.0],
            [approx_y + 1.0, approx_x + 1.0, 3.0]
        )
    )

    cy, cx, radius = result.x

    rmse = np.sqrt(np.mean(result.fun**2))

    # Check optimization and geometric fit quality
    if not result.success or rmse > 0.20:
        return {
            "status": "REVIEW",
            "reason": "Poor circle-fitting quality",
            "num_points": int(len(wall_points)),
            "fit_rmse_mm": float(rmse)
        }

    # Reject fits pressed against parameter bounds
    lower = np.array([
        approx_y - 1.0, approx_x - 1.0, 2.0
    ])

    upper = np.array([
        approx_y + 1.0, approx_x + 1.0, 3.0
    ])

    if np.any(result.x - lower < 0.01) or np.any(
        upper - result.x < 0.01
    ):
        return {
            "status": "REVIEW",
            "reason": "Circle fit reached parameter bounds",
            "num_points": int(len(wall_points))
        }

    return {
        "status": "MEASURED",
        "center_y_mm": float(cy),
        "center_x_mm": float(cx),
        "diameter_mm": float(2 * radius),
        "fit_rmse_mm": float(rmse),
        "num_points": int(len(wall_points))
    }
'''

FEATURES_PATH = PROJECT_ROOT / "src/features.py"

FEATURES_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

FEATURES_PATH.write_text(features_code)

print("Saved:", FEATURES_PATH)

Saved: /content/drive/MyDrive/Scan-to-CAD_System/src/features.py


In [28]:
import sys
import importlib

SRC_PATH = str(PROJECT_ROOT / "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import features
importlib.reload(features)

print("features.py imported successfully!")

features.py imported successfully!


In [29]:
import numpy as np

registered_points = np.asarray(
    registered_scan.points
)

hole1_config = config["features"]["hole_1"]
hole2_config = config["features"]["hole_2"]

hole1_result = features.measure_horizontal_hole(
    registered_points,
    approx_y=hole1_config["approx_center_y_mm"],
    approx_x=hole1_config["approx_center_x_mm"]
)

hole2_result = features.measure_horizontal_hole(
    registered_points,
    approx_y=hole2_config["approx_center_y_mm"],
    approx_x=hole2_config["approx_center_x_mm"]
)

print("Hole 1:")
print(hole1_result)

print("\nHole 2:")
print(hole2_result)

Hole 1:
{'status': 'MEASURED', 'center_y_mm': 15.372464979735147, 'center_x_mm': 6.428864517308433, 'diameter_mm': 4.989139899511053, 'fit_rmse_mm': 0.10190216918898293, 'num_points': 81}

Hole 2:
{'status': 'MEASURED', 'center_y_mm': 34.463490760985344, 'center_x_mm': 6.503641968558821, 'diameter_mm': 5.0162829735229595, 'fit_rmse_mm': 0.10667395958183365, 'num_points': 88}


In [30]:
bad_points = registered_points[:10]

bad_result = features.measure_horizontal_hole(
    bad_points,
    approx_y=15.5,
    approx_x=6.5
)

print(bad_result)

{'status': 'REVIEW', 'reason': 'Insufficient candidate wall points', 'num_points': 0}


In [31]:
plane_code = '''

def fit_plane_svd(points):
    """
    Fit a least-squares plane using SVD.
    Returns centroid, unit normal, and signed residuals.
    """

    points = np.asarray(points, dtype=float)

    if points.ndim != 2 or points.shape[1] != 3:
        raise ValueError("Expected an Nx3 point array.")

    if len(points) < 3:
        raise ValueError("At least 3 points are required.")

    centroid = points.mean(axis=0)
    centered = points - centroid

    _, singular_values, vh = np.linalg.svd(
        centered,
        full_matrices=False
    )

    if singular_values[1] < 1e-8:
        raise ValueError("Points do not define a unique plane.")

    normal = vh[-1]
    normal = normal / np.linalg.norm(normal)

    if normal[2] < 0:
        normal = -normal

    residuals = centered @ normal

    return centroid, normal, residuals


def measure_flange(
    nominal_points,
    scan_points,
    top_z=4.5
):
    """
    Guided horizontal-flange measurement.

    Returns REVIEW if insufficient geometry is available.
    """

    nominal_points = np.asarray(nominal_points)
    scan_points = np.asarray(scan_points)

    # Extract nominal top surface
    nominal_mask = (
        (nominal_points[:, 1] > 8.0) &
        (np.abs(nominal_points[:, 2] - top_z) < 0.05)
    )

    nominal_top = nominal_points[nominal_mask]

    if len(nominal_top) < 100:
        return {
            "status": "REVIEW",
            "reason": "Insufficient nominal top-surface points"
        }

    nominal_centroid, nominal_normal, _ = fit_plane_svd(
        nominal_top
    )

    # Estimate scan normals for guided segmentation
    import open3d as o3d

    cloud = o3d.geometry.PointCloud()
    cloud.points = o3d.utility.Vector3dVector(scan_points)

    cloud.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(
            radius=2.0,
            max_nn=40
        )
    )

    normals = np.asarray(cloud.normals)
    normal_z = np.abs(normals[:, 2])

    # Guided top-flange region
    scan_mask = (
        (scan_points[:, 0] > -1.5) &
        (scan_points[:, 0] < 14.5) &
        (scan_points[:, 1] > 8.0) &
        (scan_points[:, 1] < 49.0) &
        (scan_points[:, 2] > 3.5) &
        (scan_points[:, 2] < 9.0) &
        (normal_z > 0.90)
    )

    scan_top = scan_points[scan_mask]

    if len(scan_top) < 100:
        return {
            "status": "REVIEW",
            "reason": "Insufficient scanned flange points",
            "num_points": int(len(scan_top))
        }

    # Deviations from nominal plane
    deviations = (
        scan_top - nominal_centroid
    ) @ nominal_normal

    # Best-fit manufactured plane
    scan_centroid, scan_normal, residuals = fit_plane_svd(
        scan_top
    )

    angle = np.degrees(
        np.arccos(
            np.clip(
                np.dot(nominal_normal, scan_normal),
                -1.0,
                1.0
            )
        )
    )

    return {
        "status": "MEASURED",
        "num_points": int(len(scan_top)),
        "nominal_plane_deviation_p95_mm": float(
            np.percentile(np.abs(deviations), 95)
        ),
        "nominal_plane_deviation_max_mm": float(
            np.max(np.abs(deviations))
        ),
        "plane_fit_rmse_mm": float(
            np.sqrt(np.mean(residuals**2))
        ),
        "plane_residual_range_mm": float(
            np.ptp(residuals)
        ),
        "angular_deviation_deg": float(angle),
        "nominal_normal": nominal_normal.tolist(),
        "measured_normal": scan_normal.tolist()
    }
'''

with open(FEATURES_PATH, "a") as f:
    f.write(plane_code)

print("Plane-fitting functions added to features.py")

Plane-fitting functions added to features.py


In [32]:
import importlib
import features

importlib.reload(features)

print("Updated features.py loaded!")

Updated features.py loaded!


In [33]:
nominal_sample = nominal_mesh.sample_points_poisson_disk(
    number_of_points=30000
)

nominal_points = np.asarray(nominal_sample.points)
registered_points = np.asarray(registered_scan.points)

print("Nominal points:", len(nominal_points))
print("Registered scan points:", len(registered_points))

Nominal points: 30000
Registered scan points: 9122


In [34]:
flange_result = features.measure_flange(
    nominal_points,
    registered_points,
    top_z=4.5
)

print("Flange measurement:")

for key, value in flange_result.items():
    print(f"{key}: {value}")

Flange measurement:
status: MEASURED
num_points: 1190
nominal_plane_deviation_p95_mm: 2.2531158139381082
nominal_plane_deviation_max_mm: 4.384357700248984
plane_fit_rmse_mm: 0.463790886314049
plane_residual_range_mm: 5.451683137927268
angular_deviation_deg: 3.538270916440787
nominal_normal: [-3.756998429012873e-06, 1.1003608195239713e-06, 0.9999999999923371]
measured_normal: [0.00406832334420274, -0.06157964763785794, 0.998093881226593]


In [35]:
bad_flange_result = features.measure_flange(
    nominal_points,
    registered_points[:10],
    top_z=4.5
)

print(bad_flange_result)

{'status': 'REVIEW', 'reason': 'Insufficient scanned flange points', 'num_points': 1}


In [36]:
wrapper_code = '''

def measure_features(nominal_mesh, registered_scan, config):
    """
    Run all configured feature measurements.

    Returns structured measurements.
    """

    import numpy as np

    nominal_sample = nominal_mesh.sample_points_poisson_disk(
        number_of_points=30000
    )

    nominal_points = np.asarray(nominal_sample.points)
    scan_points = np.asarray(registered_scan.points)

    results = {
        "holes": {},
        "hole_spacing": {},
        "horizontal_flange": {}
    }

    for hole_name in ["hole_1", "hole_2"]:

        hole_cfg = config["features"][hole_name]

        approx_y = hole_cfg["approx_center_y_mm"]
        approx_x = hole_cfg["approx_center_x_mm"]

        nominal_result = measure_horizontal_hole(
            nominal_points,
            approx_y,
            approx_x
        )

        measured_result = measure_horizontal_hole(
            scan_points,
            approx_y,
            approx_x
        )

        feature_result = {
            "nominal": nominal_result,
            "measured": measured_result
        }

        if (
            nominal_result["status"] == "MEASURED"
            and measured_result["status"] == "MEASURED"
        ):
            feature_result["diameter_error_mm"] = (
                measured_result["diameter_mm"]
                - nominal_result["diameter_mm"]
            )

            dy = (
                measured_result["center_y_mm"]
                - nominal_result["center_y_mm"]
            )

            dx = (
                measured_result["center_x_mm"]
                - nominal_result["center_x_mm"]
            )

            feature_result["center_offset_2d_mm"] = float(
                np.sqrt(dx**2 + dy**2)
            )

        results["holes"][hole_name] = feature_result

    # Calculate spacing only if both holes were measured
    h1 = results["holes"]["hole_1"]
    h2 = results["holes"]["hole_2"]

    if all(
        h["nominal"]["status"] == "MEASURED"
        and h["measured"]["status"] == "MEASURED"
        for h in [h1, h2]
    ):
        def spacing(a, b):
            return float(np.hypot(
                a["center_y_mm"] - b["center_y_mm"],
                a["center_x_mm"] - b["center_x_mm"]
            ))

        nominal_spacing = spacing(
            h1["nominal"],
            h2["nominal"]
        )

        measured_spacing = spacing(
            h1["measured"],
            h2["measured"]
        )

        results["hole_spacing"] = {
            "status": "MEASURED",
            "nominal_mm": nominal_spacing,
            "measured_mm": measured_spacing,
            "error_mm": measured_spacing - nominal_spacing
        }

    else:
        results["hole_spacing"] = {
            "status": "REVIEW",
            "reason": "One or more holes could not be measured"
        }

    # Measure flange
    results["horizontal_flange"] = measure_flange(
        nominal_points,
        scan_points,
        top_z=4.5
    )

    return results
'''

with open(FEATURES_PATH, "a") as f:
    f.write(wrapper_code)

print("measure_features() added to features.py")

measure_features() added to features.py


In [37]:
import importlib
import features

importlib.reload(features)

<module 'features' from '/content/drive/MyDrive/Scan-to-CAD_System/src/features.py'>

In [38]:
measurements = features.measure_features(
    nominal_mesh,
    registered_scan,
    config
)

import json

print(json.dumps(measurements, indent=2))

{
  "holes": {
    "hole_1": {
      "nominal": {
        "status": "MEASURED",
        "center_y_mm": 15.480311761005291,
        "center_x_mm": 6.474268565176966,
        "diameter_mm": 4.9838845240566645,
        "fit_rmse_mm": 0.0034554157106248674,
        "num_points": 260
      },
      "measured": {
        "status": "MEASURED",
        "center_y_mm": 15.372464979735147,
        "center_x_mm": 6.428864517308433,
        "diameter_mm": 4.989139899511053,
        "fit_rmse_mm": 0.10190216918898293,
        "num_points": 81
      },
      "diameter_error_mm": 0.005255375454388478,
      "center_offset_2d_mm": 0.11701476741496475
    },
    "hole_2": {
      "nominal": {
        "status": "MEASURED",
        "center_y_mm": 34.54359136097863,
        "center_x_mm": 6.47423358631721,
        "diameter_mm": 4.984041196106119,
        "fit_rmse_mm": 0.00364863621111292,
        "num_points": 258
      },
      "measured": {
        "status": "MEASURED",
        "center_y_mm": 34.463490

In [39]:
for name, hole in measurements["holes"].items():

    measured = hole["measured"]

    print(f"\n{name}")

    print("Status:", measured["status"])

    if measured["status"] == "MEASURED":
        print("Diameter:", measured["diameter_mm"], "mm")
        print("Fitting RMSE:", measured["fit_rmse_mm"], "mm")

print(
    "\nFlange status:",
    measurements["horizontal_flange"]["status"]
)

print(
    "Hole spacing status:",
    measurements["hole_spacing"]["status"]
)


hole_1
Status: MEASURED
Diameter: 4.989139899511053 mm
Fitting RMSE: 0.10190216918898293 mm

hole_2
Status: MEASURED
Diameter: 5.0162829735229595 mm
Fitting RMSE: 0.10667395958183365 mm

Flange status: MEASURED
Hole spacing status: MEASURED


In [40]:
inspection_code = '''
import math


def evaluate_tolerance(error, tolerance, uncertainty=None):
    """
    Evaluate a symmetric tolerance.

    uncertainty=None means no validated uncertainty
    was supplied, so only a provisional decision
    can be made.
    """

    if error is None or not math.isfinite(error):
        return "REVIEW"

    if tolerance is None or tolerance < 0:
        return "REVIEW"

    if uncertainty is None:
        return "PASS" if abs(error) <= tolerance else "FAIL"

    if not math.isfinite(uncertainty) or uncertainty < 0:
        return "REVIEW"

    if abs(error) + uncertainty <= tolerance:
        return "PASS"

    if abs(error) - uncertainty > tolerance:
        return "FAIL"

    return "REVIEW"


def evaluate_inspection(measurements, config, registration_status):
    """
    Convert feature measurements into tolerance decisions.

    Returns an inspection report dictionary.
    """

    # Never inspect features if registration is unreliable.
    if registration_status != "ACCEPTABLE":
        return {
            "overall_status": "REVIEW",
            "reason": "Registration quality insufficient",
            "checks": []
        }

    tolerances = config["tolerances"]
    checks = []

    def add_check(name, error, tolerance, unit, quality="MEASURED"):
        if quality != "MEASURED":
            status = "REVIEW"
        else:
            status = evaluate_tolerance(error, tolerance)

        checks.append({
            "feature": name,
            "error": float(error) if error is not None else None,
            "tolerance": float(tolerance),
            "unit": unit,
            "status": status
        })

    # Hole diameter and center position
    for hole_name in ["hole_1", "hole_2"]:
        hole = measurements["holes"][hole_name]

        quality = (
            "MEASURED"
            if hole["nominal"]["status"] == "MEASURED"
            and hole["measured"]["status"] == "MEASURED"
            else "REVIEW"
        )

        add_check(
            f"{hole_name} diameter",
            hole.get("diameter_error_mm"),
            tolerances["hole_diameter_mm"],
            "mm",
            quality
        )

        add_check(
            f"{hole_name} center offset",
            hole.get("center_offset_2d_mm"),
            tolerances["hole_position_mm"],
            "mm",
            quality
        )

    # Hole spacing
    spacing = measurements["hole_spacing"]

    add_check(
        "Hole center spacing",
        spacing.get("error_mm"),
        tolerances["hole_spacing_mm"],
        "mm",
        spacing["status"]
    )

    # Flange
    flange = measurements["horizontal_flange"]

    add_check(
        "Flange nominal-plane deviation P95",
        flange.get("nominal_plane_deviation_p95_mm"),
        tolerances["flange_plane_deviation_mm"],
        "mm",
        flange["status"]
    )

    add_check(
        "Flange angular deviation",
        flange.get("angular_deviation_deg"),
        tolerances["flange_angle_deg"],
        "degrees",
        flange["status"]
    )

    statuses = [item["status"] for item in checks]

    if "FAIL" in statuses:
        overall = "FAIL"
    elif "REVIEW" in statuses:
        overall = "REVIEW"
    else:
        overall = "PASS"

    return {
        "overall_status": overall,
        "decision_basis": "provisional_demo_tolerances",
        "checks": checks,
        "summary": {
            "passed": statuses.count("PASS"),
            "failed": statuses.count("FAIL"),
            "review": statuses.count("REVIEW")
        }
    }
'''

INSPECTION_PATH = PROJECT_ROOT / "src/inspection.py"
INSPECTION_PATH.parent.mkdir(parents=True, exist_ok=True)
INSPECTION_PATH.write_text(inspection_code)

print("Saved:", INSPECTION_PATH)

Saved: /content/drive/MyDrive/Scan-to-CAD_System/src/inspection.py


In [41]:
import importlib
import inspection

importlib.reload(inspection)

results = inspection.evaluate_inspection(
    measurements,
    config,
    registration_status
)

import json
print(json.dumps(results, indent=2))

{
  "overall_status": "FAIL",
  "decision_basis": "provisional_demo_tolerances",
  "checks": [
    {
      "feature": "hole_1 diameter",
      "error": 0.005255375454388478,
      "tolerance": 0.25,
      "unit": "mm",
      "status": "PASS"
    },
    {
      "feature": "hole_1 center offset",
      "error": 0.11701476741496475,
      "tolerance": 0.5,
      "unit": "mm",
      "status": "PASS"
    },
    {
      "feature": "hole_2 diameter",
      "error": 0.0322417774168402,
      "tolerance": 0.25,
      "unit": "mm",
      "status": "PASS"
    },
    {
      "feature": "hole_2 center offset",
      "error": 0.08532853605537392,
      "tolerance": 0.5,
      "unit": "mm",
      "status": "PASS"
    },
    {
      "feature": "Hole center spacing",
      "error": 0.027892628214999604,
      "tolerance": 0.25,
      "unit": "mm",
      "status": "PASS"
    },
    {
      "feature": "Flange nominal-plane deviation P95",
      "error": 2.253053197819464,
      "tolerance": 0.5,
      "u

In [42]:
bad_registration_result = inspection.evaluate_inspection(
    measurements,
    config,
    registration_status="REVIEW"
)

print(json.dumps(bad_registration_result, indent=2))

{
  "overall_status": "REVIEW",
  "reason": "Registration quality insufficient",
  "checks": []
}


In [43]:
reporting_code = '''
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import open3d as o3d


def save_report(results, diagnostics, measurements, output_path):
    """Save a structured JSON inspection report."""

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    report = {
        "project": "3D Scan-to-CAD Dimensional Inspection",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "inspection": results,
        "registration": diagnostics,
        "measurements": measurements
    }

    with open(output_path, "w") as f:
        json.dump(report, f, indent=2, allow_nan=False)

    return report


def create_deviation_cloud(
    nominal_mesh,
    registered_scan,
    tolerance_mm=0.5,
    max_display_distance_mm=5.0
):
    """
    Color registered scan points by unsigned
    distance to nominal CAD surface.

    Green: within tolerance
    Red: above tolerance
    Gray: large-distance/unclassified points
    """

    cad_tensor = o3d.t.geometry.TriangleMesh.from_legacy(
        nominal_mesh
    )

    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(cad_tensor)

    points = np.asarray(registered_scan.points)

    distances = scene.compute_distance(
        o3d.core.Tensor(points.astype(np.float32))
    ).numpy()

    colors = np.zeros((len(points), 3))

    good = distances <= tolerance_mm

    failing = (
        (distances > tolerance_mm) &
        (distances <= max_display_distance_mm)
    )

    unclassified = distances > max_display_distance_mm

    colors[good] = [0.0, 0.8, 0.2]
    colors[failing] = [0.95, 0.1, 0.1]
    colors[unclassified] = [0.55, 0.55, 0.55]

    colored_cloud = o3d.geometry.PointCloud()

    colored_cloud.points = o3d.utility.Vector3dVector(
        points.copy()
    )

    colored_cloud.colors = o3d.utility.Vector3dVector(
        colors
    )

    summary = {
        "total_points": int(len(points)),
        "within_tolerance_points": int(good.sum()),
        "above_tolerance_points": int(failing.sum()),
        "unclassified_points": int(unclassified.sum()),
        "median_unsigned_distance_mm": float(
            np.median(distances)
        ),
        "p95_unsigned_distance_mm": float(
            np.percentile(distances, 95)
        )
    }

    return colored_cloud, summary


def save_colored_cloud(cloud, output_path):
    """Save an annotated point cloud as PLY."""

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    success = o3d.io.write_point_cloud(
        str(output_path),
        cloud
    )

    if not success:
        raise IOError(f"Could not save PLY: {output_path}")

    return str(output_path)
'''

REPORTING_PATH = PROJECT_ROOT / "src/reporting.py"

REPORTING_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORTING_PATH.write_text(reporting_code)

print("Saved:", REPORTING_PATH)

Saved: /content/drive/MyDrive/Scan-to-CAD_System/src/reporting.py


In [44]:
import importlib
import reporting

importlib.reload(reporting)

print("reporting.py loaded!")

reporting.py loaded!


In [45]:
colored_cloud, deviation_summary = (
    reporting.create_deviation_cloud(
        nominal_mesh,
        registered_scan,
        tolerance_mm=0.5
    )
)

print("Deviation summary:")

for key, value in deviation_summary.items():
    print(f"{key}: {value}")

Deviation summary:
total_points: 9122
within_tolerance_points: 7283
above_tolerance_points: 1738
unclassified_points: 101
median_unsigned_distance_mm: 0.13384485244750977
p95_unsigned_distance_mm: 1.8379746675491333


In [46]:
REPORT_PATH = (
    PROJECT_ROOT /
    "outputs/reports/inspection_report.json"
)

PLY_PATH = (
    PROJECT_ROOT /
    "outputs/visualizations/deviation_cloud.ply"
)

report = reporting.save_report(
    results,
    diagnostics,
    measurements,
    REPORT_PATH
)

saved_ply = reporting.save_colored_cloud(
    colored_cloud,
    PLY_PATH
)

print("JSON saved:", REPORT_PATH)
print("PLY saved:", saved_ply)

JSON saved: /content/drive/MyDrive/Scan-to-CAD_System/outputs/reports/inspection_report.json
PLY saved: /content/drive/MyDrive/Scan-to-CAD_System/outputs/visualizations/deviation_cloud.ply


In [47]:
import json

with open(REPORT_PATH, "r") as f:
    loaded_report = json.load(f)

loaded_cloud = o3d.io.read_point_cloud(
    str(PLY_PATH)
)

print("Overall inspection:",
      loaded_report["inspection"]["overall_status"])

print("Tolerance checks:",
      len(loaded_report["inspection"]["checks"]))

print("Saved cloud points:",
      len(loaded_cloud.points))

print("Cloud has colors:",
      loaded_cloud.has_colors())

Overall inspection: FAIL
Tolerance checks: 7
Saved cloud points: 9122
Cloud has colors: True


In [48]:
import plotly.graph_objects as go
import numpy as np

points = np.asarray(loaded_cloud.points)
colors = np.asarray(loaded_cloud.colors)

rgb_strings = [
    f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"
    for r, g, b in colors
]

fig = go.Figure(go.Scatter3d(
    x=points[:, 0],
    y=points[:, 1],
    z=points[:, 2],
    mode="markers",
    marker=dict(
        size=2,
        color=rgb_strings
    )
))

fig.update_layout(
    title="Automated Scan-to-CAD Deviation Inspection",
    scene=dict(
        xaxis_title="X (mm)",
        yaxis_title="Y (mm)",
        zaxis_title="Z (mm)",
        aspectmode="data"
    ),
    width=900,
    height=700
)

fig.show()

In [49]:
pipeline_code = '''
import json
from pathlib import Path
import numpy as np

from geometry import load_cad, load_scan
from registration import register_scan
from features import measure_features
from inspection import evaluate_inspection
from reporting import (
    save_report,
    create_deviation_cloud,
    save_colored_cloud
)


def check_registration(diagnostics, config):
    """Basic registration quality gate."""

    minimum_fitness = config["quality"][
        "minimum_registration_fitness"
    ]

    fitness = diagnostics["final_fitness"]
    rmse = diagnostics["final_rmse_mm"]

    if not np.isfinite(fitness) or fitness < minimum_fitness:
        return "REVIEW"

    if not np.isfinite(rmse):
        return "REVIEW"

    if diagnostics["stable_source_points"] < 100:
        return "REVIEW"

    T = np.asarray(diagnostics["transformation"])

    if T.shape != (4, 4) or not np.all(np.isfinite(T)):
        return "REVIEW"

    return "ACCEPTABLE"


def run_pipeline(cad_path, scan_path, config_path, output_dir):
    """
    Run the complete scan-to-CAD inspection.

    The inspection never reads synthetic ground truth.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Load configuration
    with open(config_path, "r") as f:
        config = json.load(f)

    # 2. Load inputs
    nominal_mesh = load_cad(cad_path)
    raw_scan = load_scan(scan_path)

    # 3. Registration
    registered_scan, diagnostics = register_scan(
        nominal_mesh,
        raw_scan,
        config
    )

    # 4. Registration quality
    registration_status = check_registration(
        diagnostics,
        config
    )

    # 5. Stop inspection if registration is unreliable
    if registration_status != "ACCEPTABLE":
        results = {
            "overall_status": "REVIEW",
            "reason": "Registration quality insufficient",
            "checks": [],
            "summary": {
                "passed": 0,
                "failed": 0,
                "review": 1
            }
        }

        measurements = {}

        report = save_report(
            results,
            diagnostics,
            measurements,
            output_dir / "inspection_report.json"
        )

        return report

    # 6. Feature measurements
    measurements = measure_features(
        nominal_mesh,
        registered_scan,
        config
    )

    # 7. Tolerance evaluation
    results = evaluate_inspection(
        measurements,
        config,
        registration_status
    )

    # 8. Save JSON
    report = save_report(
        results,
        diagnostics,
        measurements,
        output_dir / "inspection_report.json"
    )

    # 9. Generate and save colored point cloud
    colored_cloud, deviation_summary = create_deviation_cloud(
        nominal_mesh,
        registered_scan,
        tolerance_mm=config["tolerances"][
            "flange_plane_deviation_mm"
        ]
    )

    save_colored_cloud(
        colored_cloud,
        output_dir / "deviation_cloud.ply"
    )

    # 10. Save point-level deviation summary
    report["deviation_summary"] = deviation_summary

    with open(output_dir / "inspection_report.json", "w") as f:
        json.dump(report, f, indent=2, allow_nan=False)

    return report
'''

PIPELINE_PATH = PROJECT_ROOT / "src/run_pipeline.py"
PIPELINE_PATH.parent.mkdir(parents=True, exist_ok=True)
PIPELINE_PATH.write_text(pipeline_code)

print("Saved:", PIPELINE_PATH)

Saved: /content/drive/MyDrive/Scan-to-CAD_System/src/run_pipeline.py


In [50]:
import importlib
import run_pipeline

importlib.reload(run_pipeline)

OUTPUT_DIR = PROJECT_ROOT / "outputs/end_to_end_test"

report = run_pipeline.run_pipeline(
    cad_path=CAD_PATH,
    scan_path=SCAN_PATH,
    config_path=CONFIG_PATH,
    output_dir=OUTPUT_DIR
)

print("Overall status:", report["inspection"]["overall_status"])

print(
    "Tolerance checks:",
    len(report["inspection"]["checks"])
)

print("Outputs saved to:", OUTPUT_DIR)

[Open3D WARNING] Too few correspondences (154) after mutual filter, fall back to original correspondences.
Overall status: FAIL
Tolerance checks: 7
Outputs saved to: /content/drive/MyDrive/Scan-to-CAD_System/outputs/end_to_end_test


In [51]:
from pathlib import Path
import json
import open3d as o3d

report_path = OUTPUT_DIR / "inspection_report.json"
ply_path = OUTPUT_DIR / "deviation_cloud.ply"

print("JSON exists:", report_path.exists())
print("PLY exists:", ply_path.exists())

with open(report_path, "r") as f:
    saved_report = json.load(f)

print(
    "Overall inspection:",
    saved_report["inspection"]["overall_status"]
)

print(
    "Registration fitness:",
    saved_report["registration"]["final_fitness"]
)

if ply_path.exists():
    cloud = o3d.io.read_point_cloud(str(ply_path))

    print("Saved cloud points:", len(cloud.points))
    print("Has colors:", cloud.has_colors())

JSON exists: True
PLY exists: True
Overall inspection: FAIL
Registration fitness: 0.9868449901337426
Saved cloud points: 9122
Has colors: True


In [52]:
OUTPUT_DIR_2 = PROJECT_ROOT / "outputs/end_to_end_test_2"

report_2 = run_pipeline.run_pipeline(
    cad_path=CAD_PATH,
    scan_path=SCAN_PATH,
    config_path=CONFIG_PATH,
    output_dir=OUTPUT_DIR_2
)

print("Run 1:", report["inspection"]["overall_status"])
print("Run 2:", report_2["inspection"]["overall_status"])

print(
    "Run 1 fitness:",
    report["registration"]["final_fitness"]
)

print(
    "Run 2 fitness:",
    report_2["registration"]["final_fitness"]
)

[Open3D WARNING] Too few correspondences (167) after mutual filter, fall back to original correspondences.
Run 1: FAIL
Run 2: FAIL
Run 1 fitness: 0.9868449901337426
Run 2 fitness: 0.9868449901337426


In [53]:
import numpy as np
from scipy.spatial.transform import Rotation

T1 = np.array(
    report["registration"]["transformation"]
)

T2 = np.array(
    report_2["registration"]["transformation"]
)

# Relative rotation
R_delta = T2[:3, :3] @ T1[:3, :3].T

rotation_difference = np.degrees(
    Rotation.from_matrix(R_delta).magnitude()
)

# Translation-vector difference
translation_difference = np.linalg.norm(
    T2[:3, 3] - T1[:3, 3]
)

print(f"Rotation difference: {rotation_difference:.6f} degrees")
print(f"Translation difference: {translation_difference:.6f} mm")

Rotation difference: 0.023553 degrees
Translation difference: 0.026687 mm


In [54]:
checks_1 = report["inspection"]["checks"]
checks_2 = report_2["inspection"]["checks"]

print(f"{'Feature':38s} {'Run 1':>12s} {'Run 2':>12s} {'Difference':>12s}")

for c1, c2 in zip(checks_1, checks_2):

    assert c1["feature"] == c2["feature"]

    error1 = c1["error"]
    error2 = c2["error"]

    difference = abs(error2 - error1)

    print(
        f"{c1['feature']:38s} "
        f"{error1:12.5f} "
        f"{error2:12.5f} "
        f"{difference:12.5f}"
    )

Feature                                       Run 1        Run 2   Difference
hole_1 diameter                             0.00506      0.00514      0.00008
hole_1 center offset                        0.11636      0.11187      0.00449
hole_2 diameter                             0.02861      0.03453      0.00592
hole_2 center offset                        0.08188      0.07505      0.00684
Hole center spacing                         0.02666      0.02911      0.00245
Flange nominal-plane deviation P95          2.25302      2.25822      0.00520
Flange angular deviation                    3.54018      3.53876      0.00141


In [55]:
import json

with open(GROUND_TRUTH_PATH, "r") as f:
    ground_truth = json.load(f)

T_true = np.array(
    ground_truth["scan_pose"]["scan_to_cad_transformation"]
)

def calculate_pose_error(T_est, T_true):
    R_delta = T_est[:3, :3] @ T_true[:3, :3].T

    rotation_error = np.degrees(
        Rotation.from_matrix(R_delta).magnitude()
    )

    translation_error = np.linalg.norm(
        T_est[:3, 3] - T_true[:3, 3]
    )

    return rotation_error, translation_error


for name, T_est in [
    ("Run 1", T1),
    ("Run 2", T2)
]:
    rot, trans = calculate_pose_error(T_est, T_true)

    print(f"\n{name}")
    print(f"Rotation error: {rot:.4f} degrees")
    print(f"Translation error: {trans:.4f} mm")


Run 1
Rotation error: 0.2764 degrees
Translation error: 0.2137 mm

Run 2
Rotation error: 0.2765 degrees
Translation error: 0.1944 mm
